In [1]:
# Imports (Evidently 0.6.7)
import os, json, numpy as np, pandas as pd, evidently
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, DataQualityPreset
from evidently.metrics import ColumnDriftMetric, DatasetDriftMetric
from evidently.test_suite import TestSuite
from evidently.test_preset import DataDriftTestPreset, DataQualityTestPreset
from evidently import ColumnMapping

# Accès au paquet local `app`
import sys
PROJECT_ROOT = r"C:\Users\suean\OneDrive\Desktop\tom\OPCL2\P8"  # <-- adapte si besoin
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from app.features_optiweb import apply_eda

print("Python:", sys.executable)
print("evidently:", evidently.__version__)


Python: c:\Users\suean\OneDrive\Desktop\tom\OPCL2\P8\.venv\Scripts\python.exe
evidently: 0.6.7


## Charger X_train/X_test

In [2]:
X_train, y_train, X_test, test_ids = apply_eda(nan_as_category=True)
print("X_train:", X_train.shape, "   X_test:", X_test.shape)
X_train.head(3)



X_train: (307507, 765)    X_test: (48744, 765)


,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,...,CC_SK_DPD_DEF_VAR,CC_NAME_CONTRACT_STATUS_Active_MEAN,CC_NAME_CONTRACT_STATUS_Approved_MEAN,CC_NAME_CONTRACT_STATUS_Completed_MEAN,CC_NAME_CONTRACT_STATUS_Demand_MEAN,CC_NAME_CONTRACT_STATUS_Refused_MEAN,CC_NAME_CONTRACT_STATUS_Sent_proposal_MEAN,CC_NAME_CONTRACT_STATUS_Signed_MEAN,CC_NAME_CONTRACT_STATUS_nan_MEAN,CC_COUNT
0,0,0,0,0,202500.0,406597.5,24700.5,351000.0,0.018801,-9461,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,0,1,0,270000.0,1293502.5,35698.5,1129500.0,0.003541,-16765,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,1,0,0,67500.0,135000.0,6750.0,135000.0,0.010032,-19046,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## ColumnMapping


In [4]:
cm = ColumnMapping()
cm.numerical_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
cm.categorical_features = []         # déjà one-hot -> numérique
cm.target = None
cm.prediction = None
cm


NameError: name 'X_train' is not defined

## Rapport global (DataDrift + DataQuality)

In [5]:
os.makedirs("reports", exist_ok=True)

report_global = Report([DataDriftPreset(), DataQualityPreset()])
report_global.run(reference_data=X_train, current_data=X_test, column_mapping=cm)

report_global.save_html("reports/drift_quality_train_vs_test.html")
with open("reports/drift_quality_train_vs_test.json", "w", encoding="utf-8") as f:
    json.dump(report_global.as_dict(), f, ensure_ascii=False, indent=2)

"OK -> reports/drift_quality_train_vs_test.html"


NameError: name 'X_train' is not defined

## Métriques ciblées (DatasetDrift + 2 colonnes)

In [6]:
import json, os
from pathlib import Path
from evidently.report import Report
from evidently.metrics import DatasetDriftMetric, ColumnDriftMetric

# --- 1) Lire le JSON du rapport global ---
rep_path = Path("reports/drift_quality_train_vs_test.json")
assert rep_path.exists(), "Le JSON du rapport global est introuvable. Lance la cellule 4 d'abord."
data = json.load(open(rep_path, "r", encoding="utf-8"))

# --- 2) Extraire les colonnes avec drift_detected = True
def extract_drifted_columns(rep_dict, max_k=None):
    cols = []

    # Chemin le plus courant en 0.6.x : 'drift_by_columns' dans le preset DataDrift
    def walk(obj):
        if isinstance(obj, dict):
            if "drift_by_columns" in obj and isinstance(obj["drift_by_columns"], dict):
                return obj["drift_by_columns"]
            for v in obj.values():
                r = walk(v)
                if r is not None:
                    return r
        elif isinstance(obj, list):
            for v in obj:
                r = walk(v)
                if r is not None:
                    return r
        return None

    drift_map = walk(rep_dict) or {}
    for name, info in drift_map.items():
        try:
            if bool(info.get("drift_detected")):
                cols.append(name)
        except Exception:
            pass

    if max_k is not None:
        cols = cols[:max_k]
    return cols

# prends par ex. les 12 premières colonnes “driftées”
drift_cols = extract_drifted_columns(data, max_k=12)
print("Drifted columns (subset):", drift_cols[:10], f"... total={len(drift_cols)}")

# fallback si rien trouvé (rare) : prends 2 colonnes du dataset
if not drift_cols:
    drift_cols = X_train.columns[:2].tolist()

# --- 3) Générer un rapport ciblé uniquement sur ces colonnes ---
report_cols = Report([DatasetDriftMetric()] + [ColumnDriftMetric(c) for c in drift_cols])
report_cols.run(reference_data=X_train, current_data=X_test, column_mapping=cm)
out_html = "reports/drift_targeted_examples.html"
report_cols.save_html(out_html)
print("Saved ->", out_html)
drift_cols


AssertionError: Le JSON du rapport global est introuvable. Lance la cellule 4 d'abord.

## petit rapport maison

In [7]:
os.makedirs("reports", exist_ok=True)

# 1) récupérer les données du rapport
rep_json = Path("reports/drift_quality_train_vs_test.json")
if rep_json.exists():
    rep_dict = json.load(open(rep_json, "r", encoding="utf-8"))
else:
    # fallback si tu as encore l'objet 'report_global' en mémoire
    try:
        rep_dict = report_global.as_dict()
    except NameError:
        raise RuntimeError("Ni le JSON ni l'objet 'report_global' ne sont disponibles. Relance la cellule 4.")

# 2) helpers pour extraire ce qu'il faut (robustes aux versions 0.6.x)
def find_first(key, obj):
    if isinstance(obj, dict):
        if key in obj:
            return obj[key]
        for v in obj.values():
            r = find_first(key, v)
            if r is not None:
                return r
    elif isinstance(obj, list):
        for v in obj:
            r = find_first(key, v)
            if r is not None:
                return r
    return None

# On cherche la map des colonnes
drift_by_columns = find_first("drift_by_columns", rep_dict) or {}
# Quelques agrégats globaux si présents
number_of_columns = find_first("number_of_columns", rep_dict)
number_of_drifted = find_first("number_of_drifted_columns", rep_dict)
share_drifted     = find_first("share_of_drifted_columns", rep_dict)
dataset_drift     = find_first("dataset_drift", rep_dict)

# 3) tableau trié des colonnes (depuis Evidently)
rows = []
for name, info in drift_by_columns.items():
    rows.append({
        "column": name,
        "drift_detected": bool(info.get("drift_detected")),
        "stattest": info.get("stattest_name"),
        "drift_score": info.get("drift_score"),
        "p_value": info.get("p_value"),
        "current_small_hist": info.get("current_small_hist"),
        "ref_small_hist": info.get("ref_small_hist"),
    })
df = pd.DataFrame(rows)
df_sorted = df.sort_values(["drift_detected","drift_score"], ascending=[False, False], na_position="last")

# 4) écrire un CSV détaillé + top 15
df.to_csv("reports/drift_columns_full.csv", index=False)
top = df_sorted.head(15).copy()
top.to_csv("reports/drift_columns_top15.csv", index=False)

# 5) produire une courte conclusion
n_cols = (number_of_columns if number_of_columns is not None else len(df))
n_drift = (number_of_drifted if number_of_drifted is not None else int(df["drift_detected"].sum()))
share = (share_drifted if share_drifted is not None else (n_drift / n_cols if n_cols else 0.0))
flag = (dataset_drift if dataset_drift is not None else (share > 0.5))  # règle par défaut Evidently

lines = []
lines.append("# Drift – Conclusion rapide\n")
lines.append(f"- Colonnes totales : **{n_cols}**")
lines.append(f"- Colonnes en drift : **{n_drift}**  (part: **{share:.2%}**)  → Dataset drift **{'DETECTÉ' if flag else 'NON détecté'}**")
if not top.empty:
    lines.append("\n**Top 10 colonnes dérivées (drift_score Evidently)** :")
    for i, r in top.head(10).iterrows():
        score = r['drift_score']
        st    = r['stattest'] or "n/a"
        lines.append(f"  - `{r['column']}` — score={score:.3f} — test={st}")
# recommandations très simples
lines.append("\n**Recommandations** :")
if share >= 0.25:
    lines.append("- Part de colonnes dérivées élevée → considérer un **retrain**/recalibrage.")
elif share > 0.1:
    lines.append("- Drift modéré → **surveiller** et vérifier les perfs en ligne.")
else:
    lines.append("- Drift faible → **rien d’urgent** ; garder un œil sur les variables du top.")

summary_md = "\n".join(lines)
Path("reports/drift_summary.md").write_text(summary_md, encoding="utf-8")
print(summary_md)
print("\nFichiers écrits:",
      "reports/drift_columns_full.csv,",
      "reports/drift_columns_top15.csv,",
      "reports/drift_summary.md")


KeyError: 'drift_detected'

In [8]:
# ============================================
# Drift – focus sur les 40 features du modèle "medium"
# ============================================

from pathlib import Path

TOP40_FEATURES = [
    "PAYMENT_RATE",
    "EXT_SOURCE_3",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "DAYS_BIRTH",
    "AMT_ANNUITY",
    "APPROVED_CNT_PAYMENT_MEAN",
    "DAYS_ID_PUBLISH",
    "INSTAL_DPD_MEAN",
    "AMT_CREDIT",
    "INSTAL_AMT_PAYMENT_SUM",
    "AMT_GOODS_PRICE",
    "DAYS_EMPLOYED_PERC",
    "DAYS_REGISTRATION",
    "PREV_CNT_PAYMENT_MEAN",
    "DAYS_EMPLOYED",
    "ACTIVE_DAYS_CREDIT_MAX",
    "INSTAL_DAYS_ENTRY_PAYMENT_MAX",
    "CODE_GENDER",
    "BURO_DAYS_CREDIT_MAX",
    "ANNUITY_INCOME_PERC",
    "INCOME_CREDIT_PERC",
    "ACTIVE_DAYS_CREDIT_ENDDATE_MIN",
    "REGION_POPULATION_RELATIVE",
    "DAYS_LAST_PHONE_CHANGE",
    "ACTIVE_DAYS_CREDIT_ENDDATE_MEAN",
    "BURO_DAYS_CREDIT_ENDDATE_MAX",
    "INSTAL_PAYMENT_DIFF_MEAN",
    "PREV_APP_CREDIT_PERC_MEAN",
    "BURO_AMT_CREDIT_SUM_DEBT_MEAN",
    "BURO_AMT_CREDIT_SUM_MEAN",
    "INSTAL_DBD_SUM",
    "POS_MONTHS_BALANCE_MAX",
    "PREV_APP_CREDIT_PERC_MIN",
    "NAME_FAMILY_STATUS_Married",
    "CC_CNT_DRAWINGS_ATM_CURRENT_MEAN",
    "APPROVED_AMT_ANNUITY_MEAN",
    "INSTAL_AMT_PAYMENT_MIN",
    "INSTAL_DAYS_ENTRY_PAYMENT_MEAN",
    "APPROVED_DAYS_DECISION_MAX",
]

# On part du df déjà construit dans la cellule précédente
df_40 = df[df["column"].isin(TOP40_FEATURES)].copy()

# Tri comme avant (d’abord les colonnes en drift, puis score décroissant)
df_40_sorted = df_40.sort_values(
    ["drift_detected", "drift_score"],
    ascending=[False, False],
    na_position="last",
)

os.makedirs("reports", exist_ok=True)

# CSV complet + top 15
df_40.to_csv("reports/drift_top40_full.csv", index=False)
top40 = df_40_sorted.head(15).copy()
top40.to_csv("reports/drift_top40_top15.csv", index=False)

# Petit résumé spécifique aux 40 features
n_cols_40 = len(df_40)
n_drift_40 = int(df_40["drift_detected"].sum())
share_40 = n_drift_40 / n_cols_40 if n_cols_40 else 0.0
flag_40 = share_40 > 0.5  # règle simple : drift si > 50% des features dérivent

lines_40 = []
lines_40.append("# Drift – Focus sur les 40 features du modèle *medium*\n")
lines_40.append(f"- Features analysées : **{n_cols_40}**")
lines_40.append(
    f"- Features en drift : **{n_drift_40}**  (part: **{share_40:.2%}**) "
    f"→ Dataset drift **{'DETECTÉ' if flag_40 else 'NON détecté'}** sur ce sous-ensemble"
)

if not top40.empty:
    lines_40.append("\n**Top 10 features (drift_score Evidently)** :")
    for _, r in top40.head(10).iterrows():
        score = r["drift_score"]
        st = r["stattest"] or "n/a"
        lines_40.append(f"  - `{r['column']}` — score={score:.3f} — test={st}")

lines_40.append("\n**Recommandations (focus top40)** :")
if share_40 >= 0.25:
    lines_40.append(
        "- Part de features en drift élevée → envisager un **retrain** ou au moins un recalibrage du modèle medium."
    )
elif share_40 > 0.10:
    lines_40.append(
        "- Drift modéré → **surveiller** les perfs en prod et les features listées dans le top."
    )
else:
    lines_40.append(
        "- Drift faible → pas d’action urgente, garder un œil sur les quelques features dérivées."
    )

summary_md_40 = "\n".join(lines_40)
Path("reports/drift_top40_summary.md").write_text(summary_md_40, encoding="utf-8")
print(summary_md_40)
print(
    "\nFichiers écrits:",
    "reports/drift_top40_full.csv,",
    "reports/drift_top40_top15.csv,",
    "reports/drift_top40_summary.md",
)


KeyError: 'column'